# 🎓 Train Fake News Detection on Google Colab

**Notebook này:**
1. Clone repo từ GitHub
2. Tải dataset FakeNewsNet (wget)
3. Chuẩn bị data (train/val/test split)
4. Train model với curriculum learning
5. Lưu artifacts

In [ ]:
# 🔧 Kiểm tra GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  Bạn nên chuyển Runtime sang GPU để train nhanh hơn.')

In [ ]:
# ===== Cấu hình repo =====
REPO_URL = 'https://github.com/phidinhmanh/fake-news-detection.git'  # Đổi thành repo của bạn
BRANCH = 'main'
REPO_DIR = '/content/fake-new-detection'

# Nếu repo private: đặt USE_PRIVATE_REPO = True và điền token
USE_PRIVATE_REPO = False
GITHUB_TOKEN = ''  # ghp_xxx

# ===== Cấu hình train =====
USE_CURRICULUM = True
EPOCHS_OVERRIDE = 5
BATCH_SIZE_OVERRIDE = 16

In [ ]:
# 📥 Clone repo từ GitHub
import os
import subprocess

if USE_PRIVATE_REPO:
    if not GITHUB_TOKEN:
        raise ValueError('Repo private thì cần GITHUB_TOKEN')
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    clone_url = REPO_URL

if os.path.exists(REPO_DIR):
    print('Repo đã tồn tại, pull code mới...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True, capture_output=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True, capture_output=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True, capture_output=True)
else:
    print('Đang clone repo...')
    result = subprocess.run(
        ['git', 'clone', '-b', BRANCH, clone_url, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Clone failed:', result.stderr)
    else:
        print('Clone thành công!')

print('Sẵn sàng tại:', REPO_DIR)

In [ ]:
# 📥 Tải FakeNewsNet Dataset (sử dụng wget)
import os

%cd /content/fake-new-detection

# Tạo thư mục data/raw
!mkdir -p data/raw

# URLs của 4 file CSV từ FakeNewsNet GitHub
FAKENEWSNET_URLS = {
    'politifact_fake.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_fake.csv',
    'politifact_real.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_real.csv',
    'gossipcop_fake.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/gossipcop_fake.csv',
    'gossipcop_real.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/gossipcop_real.csv',
}

print('Đang tải FakeNewsNet dataset...')
for filename, url in FAKENEWSNET_URLS.items():
    filepath = f'data/raw/{filename}'
    if os.path.exists(filepath):
        print(f'  ✓ {filename} đã tồn tại')
    else:
        print(f'  ⏳ Tải {filename}...')
        result = subprocess.run(
            ['wget', '-q', '-O', filepath, url],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f'  ✓ {filename} tải thành công')
        else:
            print(f'  ✗ Lỗi tải {filename}: {result.stderr}')

In [ ]:
# 🧹 Chuẩn bị data: Merge và Split
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

print('Đang đọc và merge dataset...')

all_dfs = []
for filename in FAKENEWSNET_URLS.keys():
    filepath = f'data/raw/{filename}'
    df = pd.read_csv(filepath)
    
    # Gắn nhãn source và label
    parts = filename.replace('.csv', '').split('_')
    df['source'] = parts[0]  # politifact / gossipcop
    df['label'] = parts[1]   # fake / real
    df['label_binary'] = (df['label'] == 'fake').astype(int)
    
    all_dfs.append(df)
    print(f'  - {filename}: {len(df)} records')

# Merge all
df_all = pd.concat(all_dfs, ignore_index=True)
print(f'\nTổng cộng: {len(df_all)} records')

# Làm sạch text
df_all['text'] = df_all['title'].fillna('').astype(str)

# Rename columns
df_all = df_all.rename(columns={
    'id': 'news_id',
    'url': 'news_url',
})

# Split data: 80% train, 10% val, 10% test
print('\nĐang split data...')
train_df, temp_df = train_test_split(df_all, test_size=0.2, random_state=42, stratify=df_all['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f'  Train: {len(train_df)} records')
print(f'  Val:   {len(val_df)} records')
print(f'  Test:  {len(test_df)} records')

# Lưu CSV
df_all.to_csv('data/raw/fakenewsnet_clean.csv', index=False, encoding='utf-8')
train_df.to_csv('data/raw/train.csv', index=False, encoding='utf-8')
val_df.to_csv('data/raw/val.csv', index=False, encoding='utf-8')
test_df.to_csv('data/raw/test.csv', index=False, encoding='utf-8')

# Lưu Parquet (cho training nhanh hơn)
os.makedirs('data/datasets/normalized', exist_ok=True)
train_df[['text', 'label', 'label_binary', 'source']].to_parquet('data/datasets/normalized/train.parquet', index=False)
val_df[['text', 'label', 'label_binary', 'source']].to_parquet('data/datasets/normalized/val.parquet', index=False)
test_df[['text', 'label', 'label_binary', 'source']].to_parquet('data/datasets/normalized/test.parquet', index=False)

print('\n✅ Data đã chuẩn bị xong!')
print(f'  - data/raw/fakenewsnet_clean.csv')
print(f'  - data/datasets/normalized/train.parquet')
print(f'  - data/datasets/normalized/val.parquet')
print(f'  - data/datasets/normalized/test.parquet')

# 🔧 FIX LABEL MAPPING: override config LABELS để match dataset
import sys
sys.path.insert(0, '/content/fake-new-detection')
import os
os.environ['LABELS'] = 'fake,real'
print(f'\n✅ Overriding LABELS env var: {os.getenv("LABELS")}')

In [ ]:
# 📦 Cài đặt dependencies
%cd /content/fake-new-detection

# Cài dependencies cho module model
!pip -q install -r requirements-model.txt 2>/dev/null || true

# Cài thêm các thư viện cần thiết
!pip -q install pyarrow transformers peft lightning scikit-learn pandas tqdm

print('✅ Dependencies đã cài đặt!')

In [ ]:
# 🚀 TRAINING
import shlex
import subprocess

cmd = ['python', 'model/train.py', '--config', 'model/config.yaml']
if USE_CURRICULUM:
    cmd.append('--curriculum')
if EPOCHS_OVERRIDE is not None:
    cmd.extend(['--epochs', str(EPOCHS_OVERRIDE)])
if BATCH_SIZE_OVERRIDE is not None:
    cmd.extend(['--batch-size', str(BATCH_SIZE_OVERRIDE)])

print('Run command:', ' '.join(shlex.quote(x) for x in cmd))
print('='*60)
subprocess.run(cmd, check=True)

In [ ]:
# ✅ Kiểm tra artifacts
from pathlib import Path

artifacts = [
    Path('saved_models/final_model.pt'),
    Path('saved_models/lora_adapter'),
    Path('saved_models/checkpoints'),
]
for p in artifacts:
    status = '✅ OK' if p.exists() else '❌ MISSING'
    print(f'{p}: {status}')

print('\n📁 Top checkpoint files:')
checkpoints_dir = Path('saved_models/checkpoints')
if checkpoints_dir.exists():
    for ckpt in sorted(checkpoints_dir.glob('*.ckpt'))[:5]:
        print(f'  - {ckpt.name}')
else:
    print('  (No checkpoints yet)')

In [ ]:
# 💾 Optional: Lưu artifact sang Google Drive
SAVE_TO_DRIVE = False
DRIVE_TARGET = '/content/drive/MyDrive/fake-news-models'

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    import os

    drive.mount('/content/drive')
    os.makedirs(DRIVE_TARGET, exist_ok=True)
    shutil.copytree('saved_models', f'{DRIVE_TARGET}/saved_models', dirs_exist_ok=True)
    print('Đã copy saved_models ->', DRIVE_TARGET)
else:
    print('Đặt SAVE_TO_DRIVE = True nếu muốn backup lên Google Drive.')

In [ ]:
# 📦 Optional: Zip artifact để download
!zip -rq saved_models.zip saved_models
print('Đã tạo saved_models.zip')
print('(Download từ file browser bên trái)')
# from google.colab import files
# files.download('saved_models.zip')